# fact_place_rating_snapshot_poc (periodic snapshot)

Grain: one Google read of a place (`place_id`, `snapshot_ts`, `source`): `place_details` (Place Details) or `text_search` (Text Search). `rating` / `rating_count` semi-additive; `rating_change` / `rating_count_gained` = vs the previous read of the same place, any source (additive, NULL on the first read). `restaurant_id` = current `dim_restaurant` row of the place (`'unknown'` if none).

**Incremental:** reads in the window **+** reads of places still `unknown`. LAG over every read of those places, window filter after it.

Task `fact_place_rating_snapshot_poc` of the job "2. gold (poc tables)", after the dims it reads. Keys come from the `*_scd1` dims (same `xxhash64` keys as the live dims); no match → −1. Test table: the live fact and the agg tables are unchanged.

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.fact_place_rating_snapshot_poc"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.fact_place_rating_snapshot_poc"

place_details = (spark.table(f"{silver}.stg_restaurant_place_details")
                 .filter("fetch_status = 'ok' AND rating IS NOT NULL AND fetched_at IS NOT NULL")
                 .select(F.col("place_id"),
                         F.col("fetched_at").alias("snapshot_ts"),
                         F.lit("place_details").alias("source"),
                         F.col("rating").cast("double").alias("rating"),
                         F.col("rating_count").cast("int").alias("rating_count")))
text_search = (spark.table(f"{silver}.stg_places")
               .filter("rating IS NOT NULL")
               .select(F.col("place_id"),
                       F.col("snapshot_ts"),
                       F.lit("text_search").alias("source"),
                       F.col("rating").cast("double").alias("rating"),
                       F.col("rating_count").cast("int").alias("rating_count")))
restaurants = (spark.table(f"{gold}.dim_restaurant")
               .filter("is_current AND restaurant_id <> 'unknown' AND place_id IS NOT NULL")
               .groupBy("place_id").agg(F.min("restaurant_id").alias("restaurant_id")))

## One row per read

A read is `place_id` × `snapshot_ts` × `source`.

In [ ]:
rating_reads = (place_details.unionByName(text_search)
                .filter("place_id IS NOT NULL")
                .groupBy("place_id", "snapshot_ts", "source")
                .agg(F.max("rating").alias("rating"), F.max("rating_count").alias("rating_count")))

## Incremental: places read in the window + places still unknown

In [ ]:
read_day = F.to_date(F.from_utc_timestamp("snapshot_ts", TZ))
in_window = (read_day >= from_date) & (read_day < to_date)

if load_type == "incremental":
    changed_places = rating_reads.filter(in_window).select("place_id")
    unresolved_places = spark.createDataFrame([], "place_id STRING")
    if spark.catalog.tableExists(path):
        unresolved_places = spark.table(path).filter("restaurant_id = 'unknown'").select("place_id").distinct()
    rating_reads = rating_reads.join(changed_places.union(unresolved_places).distinct(), "place_id", "left_semi")

# Transformation

## Previous read of the same place

In [ ]:
from pyspark.sql.window import Window

previous_read = Window.partitionBy("place_id").orderBy("snapshot_ts", "source")
reads = (rating_reads
         .withColumn("prev_rating", F.lag("rating").over(previous_read))
         .withColumn("prev_rating_count", F.lag("rating_count").over(previous_read))
         .withColumn("prev_snapshot_ts", F.lag("snapshot_ts").over(previous_read)))

if load_type == "incremental":
    reads = (reads.filter(in_window)
             .unionByName(reads.join(unresolved_places, "place_id", "left_semi"))
             .dropDuplicates(["place_id", "snapshot_ts", "source"]))

## Join dims and select columns

In [ ]:
result_df = (
    reads.alias("read")
    .join(restaurants.alias("restaurant"), on=F.col("read.place_id") == F.col("restaurant.place_id"), how="left")
    .select(
        F.col("read.place_id").cast("string").alias("place_id"),
        F.col("read.snapshot_ts").cast("timestamp").alias("snapshot_ts"),
        F.col("read.source").cast("string").alias("source"),
        F.date_format(F.from_utc_timestamp("read.snapshot_ts", TZ), "yyyyMMdd").cast("int").alias("snapshot_date_key"),
        F.coalesce(F.col("restaurant.restaurant_id"), F.lit("unknown")).alias("restaurant_id"),
        F.col("read.rating").cast("double").alias("rating"),
        F.col("read.rating_count").cast("int").alias("rating_count"),
        F.round(F.col("read.rating") - F.col("read.prev_rating"), 2).alias("rating_change"),
        (F.col("read.rating_count") - F.col("read.prev_rating_count")).cast("int").alias("rating_count_gained"),
        F.col("read.prev_snapshot_ts").isNull().alias("is_first_snapshot"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["place_id", "snapshot_ts", "source"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        place_id STRING, snapshot_ts TIMESTAMP, source STRING, snapshot_date_key INT, restaurant_id STRING,
        rating DOUBLE, rating_count INT, rating_change DOUBLE, rating_count_gained INT, is_first_snapshot BOOLEAN,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Periodic snapshot at read grain (poc upsert method): 1 Google read of a place. rating / rating_count semi-additive, rating_change / rating_count_gained (vs previous read) additive.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: upsert (poc `upsert`) = MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new rows. Rows are never deleted.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Check: one row per grain

In [ ]:
n, grain = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT primary_checksum) FROM {path}").first()
assert n == grain, f"{path}: {n - grain} duplicate rows on (place_id, snapshot_ts, source)"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").select("operation", "operationMetrics").first()
print(f"✅ {path}: {n} rows ·", last["operation"], dict(last["operationMetrics"] or {}))